# LogTriageEnv: Training LLM Agents to Triage Production Incidents

**Meta × PyTorch × Scaler OpenEnv Grand Finale 2026**

This notebook trains an LLM agent with GRPO to identify root causes in cascading production failures.

## Quick Info
- **GPU:** T4+ required (15GB+ VRAM)
- **Time:** 10-15 minutes
- **Model:** Auto-selects 32B→7B→3B based on VRAM
- **Output:** Trained model + reward curves

## Step 1: Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("[GPU CHECK]")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {vram_gb:.1f} GB")
    VRAM_GB = vram_gb
else:
    print("No GPU found")
    VRAM_GB = 0

## Step 2: Install Dependencies

In [ ]:
print("Installing dependencies in correct order...")
print("Step 1: Upgrade pip")
!pip install -q -U pip
print("Step 2: Install Unsloth FIRST (critical for patching)")
!pip install -q unsloth
print("Step 3: Install PyTorch")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print("Step 4: Install remaining packages")
!pip install -q bitsandbytes peft trl transformers datasets accelerate matplotlib requests huggingface_hub
print("✓ All dependencies installed successfully")

## Step 3: Optional - HuggingFace Login

Skip this if you just want local training. Uncomment to push to Hub.

In [ ]:
# Optional: Uncomment to login
# from huggingface_hub import login
# login()

print("HF login: SKIPPED (model will save locally)")

## Step 4: Clone Repository

In [ ]:
import os

if not os.path.exists('logtriage-env'):
    !git clone https://github.com/rohitdecodes/logtriage-env.git
    os.chdir('logtriage-env')
else:
    os.chdir('logtriage-env')

print(f"Working dir: {os.getcwd()}")

## Step 5: The Problem

### Scenario: Production Incident at 2 AM

Six services firing alerts:
```
api-gateway      → ERROR: timeout (most visible)
auth-service     → WARN: connection pool exhausted
user-db          → ERROR: slow query
payment-db       → [no logs yet] (ROOT CAUSE - 3 hops upstream)
```

**Question:** Which service to page first?

**Naive Answer:** api-gateway ❌

**Correct Answer:** payment-db ✅

### Why It's Hard
- Root cause **never logs first**
- Symptoms cascade before causes appear
- Agent must reason **backward** through dependencies
- LLaMA 3.3 70B baseline: only 0.65 accuracy

### How We Train
GRPO with dense reward shaping forces causal reasoning:
- +0.3 for correct root cause
- +0.3 for correct escalation
- +0.3 for correct fix
- **0 for wrong combinations**

## Step 6: Intelligent Model Selection

In [ ]:
print("[MODEL SELECTION]")

if VRAM_GB >= 24:
    model_id = "Qwen/Qwen2.5-32B-Instruct"
    model_size = "32B (BEST)"
    improvement = "+0.12 to +0.15"
    print(f"✓ {VRAM_GB:.1f} GB VRAM")
    print(f"✓ Selected: {model_size}")
elif VRAM_GB >= 10:
    model_id = "Qwen/Qwen2.5-7B-Instruct"
    model_size = "7B (GOOD)"
    improvement = "+0.04 to +0.06"
    print(f"✓ {VRAM_GB:.1f} GB VRAM")
    print(f"✓ Selected: {model_size}")
else:
    model_id = "Qwen/Qwen2.5-3B-Instruct"
    model_size = "3B (FALLBACK)"
    improvement = "+0.015"
    print(f"⚠ {VRAM_GB:.1f} GB VRAM (limited)")
    print(f"⚠ Selected: {model_size}")

print()
print(f"Model: {model_id}")
print(f"Expected cascading_failure improvement: {improvement}")

## Step 7: Launch Training

⏱️ This takes ~10-15 minutes

In [ ]:
import subprocess

print("\n" + "="*60)
print("[START] LogTriageEnv Training")
print("="*60)
print(f"Model: {model_id}")
print(f"Episodes: 30 per task (90 total)")
print(f"Algorithm: GRPO + 4-bit Unsloth")
print("="*60)
print()

cmd = [
    "python", "train.py",
    "--model", model_id,
    "--task", "all",
    "--episodes", "30",
    "--load_in_4bit",
    "--grpo_max_steps", "10",
    "--env_url", "https://ogrohit-logtriage-env.hf.space"
]

try:
    subprocess.run(cmd, check=True)
    print("\n" + "="*60)
    print("✓ TRAINING COMPLETE")
    print("="*60)
except Exception as e:
    print(f"Error: {e}")

## Step 8: Analyze Results

In [ ]:
import json
import os

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print()

tasks = ["single_crash", "cascading_failure", "silent_degradation"]

for task in tasks:
    checkpoint_file = f"./phase2_checkpoints/{task}_ep25.json"
    
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            data = json.load(f)
        
        rewards = [ep.get('reward', 0) for ep in data.get('episodes', [])]
        
        if rewards:
            first_10 = sum(rewards[:10]) / 10
            last_10 = sum(rewards[-10:]) / 10
            improvement = last_10 - first_10
            
            symbol = "✓" if improvement > 0 else "↓"
            task_name = task.replace("_", " ").title()
            
            print(f"{symbol} {task_name}")
            print(f"  First 10 avg: {first_10:+.3f}")
            print(f"  Last 10 avg:  {last_10:+.3f}")
            print(f"  Improvement:  {improvement:+.3f}")
            print()

print("="*60)
print("✓ Key metric: Cascading Failure improvement")
print("  (Shows genuine multi-hop causal learning)")

## Step 9: Visualize Reward Curves

In [ ]:
import os

if os.path.exists("merge_curves.py"):
    !python merge_curves.py
    print("✓ Curves generated")
else:
    print("[INFO] merge_curves.py not found")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

if os.path.exists("reward_curve.png"):
    img = Image.open("reward_curve.png")
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Training Reward Curves", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("reward_curve.png not found")

## Step 10: Download Outputs (Colab)

In [ ]:
import os

try:
    from google.colab import files
    
    if os.path.exists("reward_curve.png"):
        print("Downloading reward_curve.png...")
        files.download("reward_curve.png")
        print("✓ Download started")
except ImportError:
    print("[INFO] Not in Colab. Files saved locally:")
    !ls -lh reward_curve.png logtriage-trained/ 2>/dev/null || echo "Check ./logtriage-trained/"

## Summary

### What You Just Did
1. ✓ Auto-selected best model for your GPU
2. ✓ Trained on 3 incident types (90 episodes total)
3. ✓ Generated reward curves
4. ✓ Produced trained agent ready for deployment

### Outputs
- `./logtriage-trained/` - Trained model
- `reward_curve.png` - Learning curves
- `./phase2_checkpoints/` - Episode data

### Next Steps
1. **Push to Hub:** `huggingface-cli login` then uncomment `--push_to_hub`
2. **Use Locally:** Load from `./logtriage-trained/`
3. **Deploy:** Integrate into on-call system

### Resources
- Environment: https://huggingface.co/spaces/OGrohit/logtriage-env
- GitHub: https://github.com/rohitdecodes/logtriage-env
- Blog: https://github.com/rohitdecodes/logtriage-env/blob/main/BLOG_POST.md